# Stage 3 Test Runner

Exercise each stage 3 endpoint (entity, franchise, keyword, metadata, award, trending, semantic, studio) in its own independently-runnable cell. Each endpoint cell runs generation (when applicable) followed by execution, printing the full LLM spec and the execution `EndpointResult`.

## Setup

Run this cell first. It imports everything each endpoint cell needs and defines shared LLM / input configuration. After this, any endpoint cell below can be run on its own.

In [1]:
import sys, os, time, asyncio, json
from datetime import date
from pathlib import Path

# Ensure project root is on the path so imports resolve.
project_root = str(Path.cwd().parent) if Path.cwd().name == "search_v2" else str(Path.cwd())
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from dotenv import load_dotenv
load_dotenv(Path(project_root) / ".env")

from implementation.llms.generic_methods import LLMProvider

# Stage 3 generation endpoints (LLM translators)
from search_v2.stage_3.entity_query_generation import generate_entity_query
from search_v2.stage_3.franchise_query_generation import generate_franchise_query
from search_v2.stage_3.keyword_query_generation import generate_keyword_query
from search_v2.stage_3.metadata_query_generation import generate_metadata_query
from search_v2.stage_3.award_query_generation import generate_award_query
from search_v2.stage_3.studio_query_generation import generate_studio_query

# Stage 3 execution endpoints (DB / Redis scorers)
from search_v2.stage_3.entity_query_execution import execute_entity_query
from search_v2.stage_3.franchise_query_execution import execute_franchise_query
from search_v2.stage_3.keyword_query_execution import execute_keyword_query
from search_v2.stage_3.metadata_query_execution import execute_metadata_query
from search_v2.stage_3.award_query_execution import execute_award_query
from search_v2.stage_3.trending_query_execution import execute_trending_query
from search_v2.stage_3.studio_query_execution import execute_studio_query
from search_v2.stage_3.semantic_query_generation import generate_semantic_preference_query, generate_semantic_dealbreaker_query
from search_v2.stage_3.semantic_query_execution import execute_semantic_dealbreaker_query, execute_semantic_preference_query

# ---- Database connections ----
# Mirrors the FastAPI lifespan handler in api/main.py: open the Postgres
# pool (created inert with open=False at import time), initialize the
# Redis async client, and ping Qdrant to fail fast if it's unreachable.
# Qdrant's AsyncQdrantClient is constructed at import time and needs no
# explicit open step — the ping here is just a readiness check.
from db.postgres import pool as postgres_pool
from db.redis import init_redis, check_redis
from db.qdrant import check_qdrant
import db.redis as _redis_module

# Idempotent: re-running the setup cell must not double-open the Postgres
# pool or leak a prior Redis client.
if postgres_pool._closed:
    await postgres_pool.open()
    print("Postgres: pool opened")
else:
    print("Postgres: pool already open")

if _redis_module._redis_client is None:
    await init_redis()
    print(f"Redis:    initialized ({await check_redis()})")
else:
    print(f"Redis:    already initialized ({await check_redis()})")

print(f"Qdrant:   {await check_qdrant()}")

/Users/michaelkeohane/Documents/movie-finder-rag/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Postgres: pool opened
Redis:    initialized (ok)
Qdrant:   ok


## Configuration

Set the LLM provider/model, today's date, and the per-endpoint sample inputs. Each endpoint cell reads from this shared config so you can tweak inputs in one place.

In [2]:
# ---- Presets (uncomment one) ----
# Each preset sets provider, model, and kwargs for no-thinking mode.

# Kimi K2.5 — no thinking
# provider, model, kwargs = LLMProvider.KIMI, "kimi-k2.5", {"enable_thinking": False}

# GPT 5.4 Mini — no thinking
provider, model, kwargs = LLMProvider.OPENAI, "gpt-5.4-mini", {"reasoning_effort": "none", "verbosity":"low"}

# Gemini 3 Flash — no thinking
# provider, model, kwargs = LLMProvider.GEMINI, "gemini-3-flash-preview", {"thinking_config": {"thinking_budget": 0}}

# ---- Or set manually ----
# provider = LLMProvider.OPENAI
# model = "gpt-5-mini"
# kwargs = {"reasoning_effort": "low"}

# ---- Today's date (used by metadata + award generators for relative-year resolution) ----
today = date.today()

# ---- Per-endpoint sample inputs ----
# Each input block mirrors what stage 2 would hand to stage 3: an
# intent_rewrite (whole-query context), a description (the single
# positive-presence requirement), and a routing_rationale (the step 2
# concept-type label that routed this item here).

# Trending has no LLM — execution-only.

print(f"Provider: {provider.value}")
print(f"Model:    {model}")
print(f"Kwargs:   {kwargs or '(defaults)'}")
print(f"Today:    {today.isoformat()}")

Provider: openai
Model:    gpt-5.4-mini
Kwargs:   {'reasoning_effort': 'none', 'verbosity': 'low'}
Today:    2026-04-21


## Result Formatting Helper

`print_scored_candidates` takes an iterable of scored objects (anything with `movie_id` and `score` attributes — e.g. `ScoredCandidate`), bulk-fetches titles and release years from `movie_card` in one query, and prints each as `score  title (year)` on its own line. Each endpoint cell below calls it after printing the raw `EndpointResult`.

In [3]:
from datetime import datetime, timezone
from typing import Iterable, Protocol

from db.postgres import fetch_movie_cards


class _Scored(Protocol):
    movie_id: int
    score: float


async def print_scored_candidates(
    candidates: Iterable[_Scored],
    *,
    limit: int | None = 25,
    sort_desc: bool = True,
) -> None:
    """Print scored movies as ``score  title (year)``, one per line.

    Accepts any iterable of objects with ``movie_id`` and ``score``
    attributes — ``ScoredCandidate`` works, but so does any duck-typed
    equivalent. Bulk-fetches card metadata in a single query (per the
    "never query Postgres per-candidate" invariant in CLAUDE.md) and
    prints score first so the column aligns visually across rows.

    Args:
        candidates: Iterable of scored movies.
        limit: Max rows to print after sorting. None prints all.
        sort_desc: Sort by score descending before printing. Disable
            to preserve the caller's ordering.
    """
    items = list(candidates)
    if not items:
        print("(no scored candidates)")
        return

    if sort_desc:
        items = sorted(items, key=lambda c: c.score, reverse=True)
    if limit is not None:
        items = items[:limit]

    cards = await fetch_movie_cards([c.movie_id for c in items])
    card_by_id = {c["movie_id"]: c for c in cards}

    for cand in items:
        card = card_by_id.get(cand.movie_id)
        if card is None:
            # Movie exists in the scored set but not in movie_card — rare,
            # but possible for stale indexes or test data. Surface it
            # instead of silently skipping.
            title, year = "<missing card>", "?"
        else:
            title = card["title"] or "<untitled>"
            ts = card["release_ts"]
            year = (
                datetime.fromtimestamp(ts, tz=timezone.utc).year
                if ts is not None else "?"
            )
        print(f"  {cand.score:.3f}  {title} ({year})")

## Endpoint 1: Entity

Translates a named-entity lookup (person / character / studio / title pattern) into an `EntityQuerySpec` and executes it against the lexical posting tables.

In [7]:
entity_inputs = {
    "intent_rewrite": "Joker movies",
    "description": "has the joker as a character",
    "routing_rationale": "named person - character",
}

# Candidate pool (tmdb_ids). Empty set → run unrestricted; populate to exercise the pool-scoring path.
entity_candidate_ids: set[int] = set() # {11, 12, 13, 1034541, 603, 27205, 155, 680, 157336}

# ---- Generation ----
start = time.perf_counter()
entity_spec, entity_in_tok, entity_out_tok = await generate_entity_query(
    intent_rewrite=entity_inputs["intent_rewrite"],
    description=entity_inputs["description"],
    routing_rationale=entity_inputs["routing_rationale"],
    provider=provider,
    model=model,
    **kwargs,
)
entity_gen_elapsed = time.perf_counter() - start

print(f"Generation completed in {entity_gen_elapsed:.2f}s")
print(f"Tokens — input: {entity_in_tok:,}  output: {entity_out_tok:,}\n")
print("=" * 60)
print("LLM RESULT — EntityQuerySpec")
print("=" * 60)
print(entity_spec.model_dump_json(indent=2))

# ---- Execution ----
restrict = entity_candidate_ids or None
start = time.perf_counter()
entity_result = await execute_entity_query(entity_spec, restrict_to_movie_ids=restrict)
entity_exec_elapsed = time.perf_counter() - start

print()
print("=" * 60)
print(f"EXECUTION RESULT [{'pool' if restrict else 'unrestricted'}] — {len(entity_result.scores)} movies in {entity_exec_elapsed:.2f}s")
print("=" * 60)
await print_scored_candidates(entity_result.scores)

Generation completed in 3.91s
Tokens — input: 3,537  output: 119

LLM RESULT — EntityQuerySpec
{
  "entity_type_evidence": "Character lookup: the description says the film has the Joker as a character; routing_rationale confirms character.",
  "name_resolution_notes": "primary credited form",
  "lookup_text": "The Joker",
  "entity_type": "character",
  "person_category": null,
  "primary_category": null,
  "prominence_evidence": null,
  "actor_prominence_mode": null,
  "title_pattern_match_type": null,
  "character_alternative_names": [],
  "character_prominence_evidence": "no prominence signal; described as \"has the joker as a character\"",
  "character_prominence_mode": "default"
}

EXECUTION RESULT [unrestricted] — 26 movies in 0.32s
  0.979  Batman Ninja (2018)
  0.969  Batman: The Killing Joke (2016)
  0.958  Batman Unlimited: Monster Mayhem (2015)
  0.957  Justice League: Crisis on Infinite Earths Part Three (2024)
  0.955  Batman (1966)
  0.950  Grayson (2004)
  0.940  Suicide

## Endpoint 2: Franchise

Translates a franchise / shared-universe requirement into a `FranchiseQuerySpec` and executes it against `movie_franchise_metadata`.

In [5]:
franchise_inputs = {
    "intent_rewrite": "shrek movies",
    "description": "is part of the shrek universe but lineage is preferred",
    "routing_rationale": "named franchise / shared universe",
}

# Candidate pool (tmdb_ids). Empty set → run unrestricted; populate to exercise the pool-scoring path.
franchise_candidate_ids: set[int] = set() # {11, 12, 13, 1034541, 603, 27205, 155, 680, 157336}

# ---- Generation ----
start = time.perf_counter()
franchise_spec, franchise_in_tok, franchise_out_tok = await generate_franchise_query(
    intent_rewrite=franchise_inputs["intent_rewrite"],
    description=franchise_inputs["description"],
    routing_rationale=franchise_inputs["routing_rationale"],
    provider=provider,
    model=model,
    **kwargs,
)
franchise_gen_elapsed = time.perf_counter() - start

print(f"Generation completed in {franchise_gen_elapsed:.2f}s")
print(f"Tokens — input: {franchise_in_tok:,}  output: {franchise_out_tok:,}\n")
print("=" * 60)
print("LLM RESULT — FranchiseQuerySpec")
print("=" * 60)
print(franchise_spec.model_dump_json(indent=2))

# ---- Execution ----
restrict = franchise_candidate_ids or None
start = time.perf_counter()
franchise_result = await execute_franchise_query(franchise_spec, restrict_to_movie_ids=restrict)
franchise_exec_elapsed = time.perf_counter() - start

print()
print("=" * 60)
print(f"EXECUTION RESULT [{'pool' if restrict else 'unrestricted'}] — {len(franchise_result.scores)} movies in {franchise_exec_elapsed:.2f}s")
print("=" * 60)
await print_scored_candidates(franchise_result.scores)

Generation completed in 2.70s
Tokens — input: 3,384  output: 166

LLM RESULT — FranchiseQuerySpec
{
  "concept_analysis": "Scope: specific lineage inside an umbrella. The description says \"part of the shrek universe\" and the intent_rewrite is \"shrek movies,\" which signals the named franchise/universe axis. No named subgroup is mentioned, no sequel/prequel/remake/reboot wording appears, no spinoff/crossover phrasing appears, and no launch behavior is requested. Because this is exactly one specific franchise with known spinoffs/universe-adjacent entries, and the wording does not explicitly invite those entries, prefer_lineage=true.",
  "franchise_or_universe_names": [
    "shrek"
  ],
  "recognized_subgroups": null,
  "lineage_position": null,
  "structural_flags": null,
  "launch_scope": null,
  "prefer_lineage": true
}

EXECUTION RESULT [unrestricted] — 15 movies in 0.08s
  1.000  Shrek the Musical (2013)
  1.000  The Pig Who Cried Werewolf (2011)
  1.000  Shrek (2001)
  1.000  Shr

## Endpoint 3: Keyword

Translates a classification / tag requirement into a `KeywordQuerySpec` (one `UnifiedClassification` registry pick) and executes the backing posting-list lookup.

In [8]:
keyword_inputs = {
    "intent_rewrite": "Find animated movies for kids",
    "description": "is an animated movie",
    "routing_rationale": "genre classification",
}

# Candidate pool (tmdb_ids). Empty set → run unrestricted; populate to exercise the pool-scoring path.
keyword_candidate_ids: set[int] = {11, 12, 13, 1034541, 603, 27205, 155, 680, 157336}

# ---- Generation ----
start = time.perf_counter()
keyword_spec, keyword_in_tok, keyword_out_tok = await generate_keyword_query(
    intent_rewrite=keyword_inputs["intent_rewrite"],
    description=keyword_inputs["description"],
    routing_rationale=keyword_inputs["routing_rationale"],
    provider=provider,
    model=model,
    **kwargs,
)
keyword_gen_elapsed = time.perf_counter() - start

print(f"Generation completed in {keyword_gen_elapsed:.2f}s")
print(f"Tokens — input: {keyword_in_tok:,}  output: {keyword_out_tok:,}\n")
print("=" * 60)
print("LLM RESULT — KeywordQuerySpec")
print("=" * 60)
print(keyword_spec.model_dump_json(indent=2))

# ---- Execution ----
restrict = keyword_candidate_ids or None
start = time.perf_counter()
keyword_result = await execute_keyword_query(keyword_spec, restrict_to_movie_ids=restrict)
keyword_exec_elapsed = time.perf_counter() - start

print()
print("=" * 60)
print(f"EXECUTION RESULT [{'pool' if restrict else 'unrestricted'}] — {len(keyword_result.scores)} movies in {keyword_exec_elapsed:.2f}s")
print("=" * 60)
await print_scored_candidates(keyword_result.scores)

Generation completed in 1.25s
Tokens — input: 10,865  output: 60

LLM RESULT — KeywordQuerySpec
{
  "concept_analysis": "\"animated\" → Animation family; \"for kids\" → audience/life-stage boundary, but no specific family label beyond family-friendly intent",
  "candidate_shortlist": "ANIMATION: told through animated moving images — present",
  "classification": "ANIMATION"
}

EXECUTION RESULT [pool] — 9 movies in 0.36s
  1.000  Finding Nemo (2003)
  0.000  Inception (2010)
  0.000  Pulp Fiction (1994)
  0.000  Star Wars (1977)
  0.000  Terrifier 3 (2024)
  0.000  Forrest Gump (1994)
  0.000  Interstellar (2014)
  0.000  The Matrix (1999)
  0.000  The Dark Knight (2008)


## Endpoint 4: Metadata

Translates a structured-attribute requirement (date, runtime, maturity, streaming, language, country, budget, box office, popularity, reception) into a `MetadataTranslationOutput` and executes the scoring query against `movie_card`.

In [14]:
metadata_inputs = {
    "intent_rewrite": "I am looking for low-rated or poorly reviewed movies starring Seth Rogen that are enjoyable to watch while under the influence of cannabis.",
    "description": "low-rated or poorly reviewed",
    "routing_rationale": "quantitative reception score",
}

# Candidate pool (tmdb_ids). Empty set → run unrestricted; populate to exercise the pool-scoring path.
metadata_candidate_ids: set[int] = set() # {11, 12, 13, 1034541, 603, 27205, 155, 680, 157336}

# ---- Generation ----
start = time.perf_counter()
metadata_spec, metadata_in_tok, metadata_out_tok = await generate_metadata_query(
    intent_rewrite=metadata_inputs["intent_rewrite"],
    description=metadata_inputs["description"],
    routing_rationale=metadata_inputs["routing_rationale"],
    today=today,
    provider=provider,
    model=model,
    **kwargs,
)
metadata_gen_elapsed = time.perf_counter() - start

print(f"Generation completed in {metadata_gen_elapsed:.2f}s")
print(f"Tokens — input: {metadata_in_tok:,}  output: {metadata_out_tok:,}\n")
print("=" * 60)
print("LLM RESULT — MetadataTranslationOutput")
print("=" * 60)
print(metadata_spec.model_dump_json(indent=2))

# ---- Execution ----
restrict = metadata_candidate_ids or None
start = time.perf_counter()
metadata_result = await execute_metadata_query(metadata_spec, restrict_to_movie_ids=restrict)
metadata_exec_elapsed = time.perf_counter() - start

print()
print("=" * 60)
print(f"EXECUTION RESULT [{'pool' if restrict else 'unrestricted'}] — {len(metadata_result.scores)} movies in {metadata_exec_elapsed:.2f}s")
print("=" * 60)
await print_scored_candidates(metadata_result.scores)

Generation completed in 2.57s
Tokens — input: 6,690  output: 91

LLM RESULT — MetadataTranslationOutput
{
  "constraint_phrases": [
    "low-rated",
    "poorly reviewed"
  ],
  "target_attribute": "reception",
  "value_intent_label": "poorly received",
  "release_date": null,
  "runtime": null,
  "maturity_rating": null,
  "streaming": null,
  "audio_language": null,
  "country_of_origin": null,
  "budget_scale": null,
  "box_office": null,
  "popularity": null,
  "reception": "poorly_received"
}

EXECUTION RESULT [unrestricted] — 5000 movies in 4.54s
  1.000  United Passions (2014)
  1.000  Melania (2026)
  1.000  Merah Putih: One for All (2025)
  1.000  2025: The World Enslaved by a Virus (2021)
  0.975  What's Up: Balloon to the Rescue! (2009)
  0.975  Saving the Gorillas: Ellen's Next Adventure (2023)
  0.965  Superbabies: Baby Geniuses 2 (2004)
  0.965  InAPPropriate Comedy (2013)
  0.965  The Garbage Pail Kids Movie (1987)
  0.950  Sadak 2 (2020)
  0.950  Deshdrohi (2008)
  0.95

## Endpoint 5: Award

Translates an award / prestige requirement into an `AwardQuerySpec` and executes it against `movie_awards` (with the `award_ceremony_win_ids` fast path when applicable).

In [ ]:
award_inputs = {
    "intent_rewrite": "Find movies that won at least one Oscar",
    "description": "won at least one oscar",
    "routing_rationale": "award / prestige recognition",
}

# Candidate pool (tmdb_ids). Empty set → run unrestricted; populate to exercise the pool-scoring path.
award_candidate_ids: set[int] = {11, 12, 13, 1034541, 603, 27205, 155, 680, 157336}

# ---- Generation ----
start = time.perf_counter()
award_spec, award_in_tok, award_out_tok = await generate_award_query(
    intent_rewrite=award_inputs["intent_rewrite"],
    description=award_inputs["description"],
    routing_rationale=award_inputs["routing_rationale"],
    today=today,
    provider=provider,
    model=model,
    **kwargs,
)
award_gen_elapsed = time.perf_counter() - start

print(f"Generation completed in {award_gen_elapsed:.2f}s")
print(f"Tokens — input: {award_in_tok:,}  output: {award_out_tok:,}\n")
print("=" * 60)
print("LLM RESULT — AwardQuerySpec")
print("=" * 60)
print(award_spec.model_dump_json(indent=2))

# ---- Execution ----
restrict = award_candidate_ids or None
start = time.perf_counter()
award_result = await execute_award_query(award_spec, restrict_to_movie_ids=restrict)
award_exec_elapsed = time.perf_counter() - start

print()
print("=" * 60)
print(f"EXECUTION RESULT [{'pool' if restrict else 'unrestricted'}] — {len(award_result.scores)} movies in {award_exec_elapsed:.2f}s")
print("=" * 60)
await print_scored_candidates(award_result.scores)

Generation completed in 2.65s
Tokens — input: 6,911  output: 124

LLM RESULT — AwardQuerySpec
{
  "concept_analysis": "ceremony: \"oscar\" signals the Academy Awards; award name: \"oscar\" signals the specific prize object; category: no category signal; outcome: \"won\" signals winner; year: no year signal; count: \"at least one\" indicates a hard minimum count.",
  "scoring_shape_label": "specific filter, no count",
  "scoring_mode": "floor",
  "scoring_mark": 1,
  "ceremonies": [
    "Academy Awards, USA"
  ],
  "award_names": [
    "Oscar"
  ],
  "category_tags": null,
  "outcome": "winner",
  "years": null
}

EXECUTION RESULT [pool] — 9 movies in 0.61s
  1.000  Inception (2010)
  1.000  Pulp Fiction (1994)
  1.000  Star Wars (1977)
  1.000  Finding Nemo (2003)
  1.000  Forrest Gump (1994)
  1.000  Interstellar (2014)
  1.000  The Matrix (1999)
  1.000  The Dark Knight (2008)
  0.000  Terrifier 3 (2024)


## Endpoint 6: Trending

Trending has no LLM translator — stage 2 flags the intent and execution reads the precomputed trending hash from Redis directly.

In [11]:
# Candidate pool (tmdb_ids). Empty set → run unrestricted; populate to exercise the pool-scoring path.
trending_candidate_ids: set[int] = {11, 12, 13, 1034541, 603, 27205, 155, 680, 157336}

# ---- Execution only (no generation step) ----
restrict = trending_candidate_ids or None
start = time.perf_counter()
trending_result = await execute_trending_query(restrict_to_movie_ids=restrict)
trending_exec_elapsed = time.perf_counter() - start

print("=" * 60)
print("LLM RESULT — (none; trending has no LLM translator)")
print("=" * 60)
print()
print("=" * 60)
print(f"EXECUTION RESULT [{'pool' if restrict else 'unrestricted'}] — {len(trending_result.scores)} movies in {trending_exec_elapsed:.2f}s")
print("=" * 60)
await print_scored_candidates(trending_result.scores)

Redis key 'dev:trending:current' is absent or empty — trending scores unavailable


LLM RESULT — (none; trending has no LLM translator)

EXECUTION RESULT [pool] — 9 movies in 0.03s
  0.000  Inception (2010)
  0.000  Pulp Fiction (1994)
  0.000  Star Wars (1977)
  0.000  Finding Nemo (2003)
  0.000  Terrifier 3 (2024)
  0.000  Forrest Gump (1994)
  0.000  Interstellar (2014)
  0.000  The Matrix (1999)
  0.000  The Dark Knight (2008)


## Endpoint 7a: Semantic — Dealbreaker Flow

Translates a single positive-presence semantic trait into a `SemanticDealbreakerSpec` — one LLM call picks exactly one of 7 non-anchor vector spaces and emits a structured-label body. Generation only; no execution endpoint exists yet.

In [ ]:
from db.qdrant import qdrant_client

semantic_dealbreaker_inputs = {
    "intent_rewrite": "Find slow-burn psychological thrillers with an unreliable narrator",
    "description": "features an unreliable narrator as a central storytelling device",
    "routing_rationale": "narrative craft / storytelling technique",
}

# Candidate pool of tmdb_ids (D1). Empty set → D2 (probe generates its own pool).
semantic_dealbreaker_candidate_ids: set[int] = {11, 12, 13, 1034541, 603, 27205, 155, 680, 157336}

# ---- Generation ----
start = time.perf_counter()
semantic_db_spec, semantic_db_in_tok, semantic_db_out_tok = await generate_semantic_dealbreaker_query(
    intent_rewrite=semantic_dealbreaker_inputs["intent_rewrite"],
    description=semantic_dealbreaker_inputs["description"],
    routing_rationale=semantic_dealbreaker_inputs["routing_rationale"],
    provider=provider,
    model=model,
    **kwargs,
)
semantic_db_gen_elapsed = time.perf_counter() - start

print(f"Generation completed in {semantic_db_gen_elapsed:.2f}s")
print(f"Tokens — input: {semantic_db_in_tok:,}  output: {semantic_db_out_tok:,}\n")
print("=" * 60)
print("LLM RESULT — SemanticDealbreakerSpec")
print("=" * 60)
print(semantic_db_spec.model_dump_json(indent=2))

# ---- Embedding text (what actually gets vectorized) ----
print()
print("=" * 60)
print(f"EMBEDDING TEXT — space={semantic_db_spec.body.space.value}")
print("=" * 60)
print(semantic_db_spec.body.content.embedding_text())

# ---- Execution ----
restrict = semantic_dealbreaker_candidate_ids or None
start = time.perf_counter()
semantic_db_result = await execute_semantic_dealbreaker_query(
    semantic_db_spec,
    restrict_to_movie_ids=restrict,
    qdrant_client=qdrant_client,
)
semantic_db_exec_elapsed = time.perf_counter() - start

scenario = "D1 (pre-built pool)" if restrict else "D2 (probe generates candidates)"
print()
print("=" * 60)
print(f"EXECUTION RESULT [{scenario}] — {len(semantic_db_result.scores)} movies in {semantic_db_exec_elapsed:.2f}s")
print("=" * 60)
await print_scored_candidates(semantic_db_result.scores)

## Endpoint 7b: Semantic — Preference Flow

Translates a grouped preference description into a `SemanticPreferenceSpec` — decomposes into atomic qualifiers, picks 1+ of the 8 vector spaces (anchor allowed), assigns primary/contributing weights, and emits one body per selected space. Generation only; no execution endpoint exists yet.

In [ ]:
from db.qdrant import qdrant_client

semantic_preference_inputs = {
    "intent_rewrite": "Something slow-burn, atmospheric, and melancholy for a rainy Sunday afternoon",
    "description": "slow-burn, atmospheric, rainy-day melancholy for a quiet Sunday afternoon",
    "routing_rationale": "viewing mood / tone preference",
}

# Candidate pool of tmdb_ids (P1). Empty set → P2 (per-space probes generate pool).
semantic_preference_candidate_ids: set[int] = {11, 12, 13, 1034541, 603, 27205, 155, 680, 157336}

# ---- Generation ----
start = time.perf_counter()
semantic_pref_spec, semantic_pref_in_tok, semantic_pref_out_tok = await generate_semantic_preference_query(
    intent_rewrite=semantic_preference_inputs["intent_rewrite"],
    description=semantic_preference_inputs["description"],
    routing_rationale=semantic_preference_inputs["routing_rationale"],
    provider=provider,
    model=model,
    **kwargs,
)
semantic_pref_gen_elapsed = time.perf_counter() - start

print(f"Generation completed in {semantic_pref_gen_elapsed:.2f}s")
print(f"Tokens — input: {semantic_pref_in_tok:,}  output: {semantic_pref_out_tok:,}\n")
print("=" * 60)
print("LLM RESULT — SemanticPreferenceSpec")
print("=" * 60)
print(semantic_pref_spec.model_dump_json(indent=2))

# ---- Embedding texts per selected space (what actually gets vectorized) ----
print()
print("=" * 60)
print("EMBEDDING TEXTS (one per selected space)")
print("=" * 60)
for entry in semantic_pref_spec.space_queries:
    print(f"\n--- space={entry.space.value}  weight={entry.weight.value} ---")
    print(entry.content.embedding_text())

# ---- Execution ----
restrict = semantic_preference_candidate_ids or None
start = time.perf_counter()
semantic_pref_result = await execute_semantic_preference_query(
    semantic_pref_spec,
    restrict_to_movie_ids=restrict,
    qdrant_client=qdrant_client,
)
semantic_pref_exec_elapsed = time.perf_counter() - start

scenario = "P1 (pre-built pool)" if restrict else "P2 (probes generate candidates)"
print()
print("=" * 60)
print(f"EXECUTION RESULT [{scenario}] — {len(semantic_pref_result.scores)} movies in {semantic_pref_exec_elapsed:.2f}s")
print("=" * 60)
await print_scored_candidates(semantic_pref_result.scores)

## Endpoint 8: Studio

Translates a studio / production-company lookup into a `StudioQuerySpec` (either a curated `brand_id` for umbrella queries, or up to 3 `freeform_names` for sub-labels / long-tail studios) and executes it against `lex.inv_production_brand_postings` (brand path) or the `lex.studio_token` intersection index (freeform path).

In [6]:
studio_inputs = {
    "intent_rewrite": "Find Disney films",
    "description": "produced by Disney",
    "routing_rationale": "named production company / studio",
}

# Candidate pool (tmdb_ids). Empty set → run unrestricted; populate to exercise the pool-scoring path.
studio_candidate_ids: set[int] = set() # {11, 12, 13, 1034541, 603, 27205, 155, 680, 157336}

# ---- Generation ----
start = time.perf_counter()
studio_spec, studio_in_tok, studio_out_tok = await generate_studio_query(
    intent_rewrite=studio_inputs["intent_rewrite"],
    description=studio_inputs["description"],
    routing_rationale=studio_inputs["routing_rationale"],
    provider=provider,
    model=model,
    **kwargs,
)
studio_gen_elapsed = time.perf_counter() - start

print(f"Generation completed in {studio_gen_elapsed:.2f}s")
print(f"Tokens — input: {studio_in_tok:,}  output: {studio_out_tok:,}\n")
print("=" * 60)
print("LLM RESULT — StudioQuerySpec")
print("=" * 60)
print(studio_spec.model_dump_json(indent=2))

# ---- Execution ----
restrict = studio_candidate_ids or None
start = time.perf_counter()
studio_result = await execute_studio_query(studio_spec, restrict_to_movie_ids=restrict)
studio_exec_elapsed = time.perf_counter() - start

print()
print("=" * 60)
print(f"EXECUTION RESULT [{'pool' if restrict else 'unrestricted'}] — {len(studio_result.scores)} movies in {studio_exec_elapsed:.2f}s")
print("=" * 60)
await print_scored_candidates(studio_result.scores)

Generation completed in 2.31s
Tokens — input: 3,084  output: 45

LLM RESULT — StudioQuerySpec
{
  "thinking": "This is an umbrella/parent-brand query; the registry covers it. I will emit the Disney brand slug.",
  "brand_id": "disney",
  "freeform_names": null
}

EXECUTION RESULT [unrestricted] — 1838 movies in 0.08s
  1.000  Four Rooms (1995)
  1.000  The Quiet American (2002)
  1.000  Winnie the Pooh: Springtime with Roo (2004)
  1.000  Finding Nemo (2003)
  1.000  Dug's Special Mission (2009)
  1.000  Pirates of the Caribbean: The Curse of the Black Pearl (2003)
  1.000  Kill Bill: Vol. 1 (2003)
  1.000  Of Love and Shadows (1994)
  1.000  King of the Grizzlies (1970)
  1.000  A Month by the Lake (1995)
  1.000  How to Ride a Horse (1947)
  1.000  Pirates of the Caribbean: Dead Man's Chest (2006)
  1.000  Unidentified Flying Oddball (1979)
  1.000  The Crow: Wicked Prayer (2005)
  1.000  The Lookout (2007)
  1.000  Trevor: The Musical (2022)
  1.000  Armageddon (1998)
  1.000  Tron 